In [1]:
import cv2
from imutils.video import VideoStream
import datetime
import time

# Minimum contour area
min_area = 500

# Start webcam
vs = VideoStream(src=0).start()
time.sleep(2.0)

firstFrame = None

while True:

    frame = vs.read()

    if frame is None:
        break

    # Resize frame
    frame = cv2.resize(frame, (500, 400))

    # Convert to grayscale
    gray = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2GRAY
    )

    # Blur image
    gray = cv2.GaussianBlur(
        gray,
        (21, 21),
        0
    )

    # Save first frame as background
    if firstFrame is None:
        firstFrame = gray
        continue

    # Difference between current frame and background
    frameDelta = cv2.absdiff(
        firstFrame,
        gray
    )

    # Threshold
    thresh = cv2.threshold(
        frameDelta,
        25,
        255,
        cv2.THRESH_BINARY
    )[1]

    # Dilate image
    thresh = cv2.dilate(
        thresh,
        None,
        iterations=2
    )

    # Find contours
    cnts, _ = cv2.findContours(
        thresh.copy(),
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    text = "Unoccupied"

    for c in cnts:

        if cv2.contourArea(c) < min_area:
            continue

        (x, y, w, h) = cv2.boundingRect(c)

        cv2.rectangle(
            frame,
            (x, y),
            (x + w, y + h),
            (0, 255, 0),
            2
        )

        text = "Occupied"

    # Status text
    cv2.putText(
        frame,
        f"Room Status: {text}",
        (10, 20),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (0, 0, 255),
        2
    )

    # Date and Time
    cv2.putText(
        frame,
        datetime.datetime.now().strftime(
            "%d/%m/%Y %H:%M:%S"
        ),
        (10, frame.shape[0] - 10),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        (0, 0, 255),
        1
    )

    cv2.imshow(
        "Security Feed",
        frame
    )

    cv2.imshow(
        "Threshold",
        thresh
    )

    cv2.imshow(
        "Frame Delta",
        frameDelta
    )

    key = cv2.waitKey(1) & 0xFF

    if key == ord("q"):
        break

vs.stop()
cv2.destroyAllWindows()